# 05. Семантический baseline

Этот ноутбук строит baseline-модель для задачи **Semantic News Novelty**.

Baseline оформлен как переносимый модельный пайплайн:

1. принимает на вход данные в формате, аналогичном `lenta_clean_news.csv`;
2. при необходимости приводит файлы разметки/темплейты в новой простой схеме к clean-like формату;
3. строит sentence embeddings с помощью `BAAI/bge-m3`;
4. сам формирует `cluster_id`;
5. сам формирует `novelty_label` внутри кластеров;
6. сохраняет prediction-файл в формате, совместимом с `04_evaluate_annotations_and_predictions.ipynb`.

Выход baseline:

```text
news_id
published_at
topic
title
text
cluster_id
novelty_label
comment
needs_review
```

Важно: если на вход подан файл разметки, baseline **не использует** входные `cluster_id`, `novelty_label`, `comment`, `needs_review`. Эти поля считаются выводом другой модели/аннотатора и игнорируются.


## 1. Импорты


In [38]:
from __future__ import annotations

import hashlib
import json
import math
import os
import pickle
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    adjusted_rand_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)

## 2. Конфигурация

`INPUT_PATH` может указывать как на clean-like файл, так и на файл в новой простой схеме разметки.

Clean-like формат аналогичен `lenta_clean_news.csv`:

```text
news_id, url, title, text, topic, tags, published_at,
text_length, text_num_words, title_length, title_num_words,
model_text, model_length, model_num_words
```

Новая простая схема разметки/темплейта:

```text
news_id, published_at, topic, title, text,
cluster_id, novelty_label, comment, needs_review
```

`FORCE_REDOWNLOAD_MODEL` нужен только если требуется заново скачать/сохранить sentence-transformer.

`FORCE_RECOMPUTE_EMBEDDINGS` нужен, если изменился входной датасет, колонка `model_text`, модель embeddings или способ подготовки текста.

Если меняются только пороги `STORY_THRESHOLD`, `MINOR_THRESHOLD`, `DUPLICATE_THRESHOLD` или временное окно, **переcохранять embedding-модель и пересчитывать embeddings не нужно**. Нужно только заново построить кластеры и predictions.


In [39]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared"
BASELINE_DIR = DATA_DIR / "baseline"

ARTIFACTS_DIR = DATA_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_ARTIFACTS_DIR = ARTIFACTS_DIR / "baseline"
BASELINE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Основной вход.
# Рекомендуемый вариант для end-to-end baseline: полный candidate pool без выводов модели.
# Для быстрой проверки можно указать файл в простой схеме разметки; поля разметки будут проигнорированы.
INPUT_PATH = PREPARED_DIR / "lenta_golden_candidate_pool.csv"

# Опциональный reference для быстрой оценки прямо внутри этого ноутбука.
# Обычно здесь указывается human gold.
REFERENCE_PATH = PREPARED_DIR / "lenta_golden_set_human.csv"
RUN_INLINE_EVALUATION = True

OUTPUT_PATH = PREPARED_DIR / "lenta_baseline_predictions.csv"
SUMMARY_PATH = BASELINE_ARTIFACTS_DIR / "lenta_baseline_summary.json"

MODEL_NAME = "BAAI/bge-m3"
LOCAL_MODEL_DIR = BASELINE_ARTIFACTS_DIR / "sentence_transformer_bge_m3"
EMBEDDINGS_CACHE_DIR = BASELINE_ARTIFACTS_DIR / "embeddings_cache"
EMBEDDINGS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REDOWNLOAD_MODEL = False
FORCE_RECOMPUTE_EMBEDDINGS = False
FORCE_RERUN_BASELINE = True

# Параметры кластеризации baseline.
STORY_THRESHOLD = 0.82
STORY_WINDOW_DAYS = 14
# Параметры rule-based novelty-разметки.
MINOR_THRESHOLD = 0.88
DUPLICATE_THRESHOLD = 0.90
REVIEW_MARGIN = 0.03

# Простое topic-wise ограничение: новости кластеризуются внутри одной рубрики.
USE_TOPIC_BLOCKING = True

# Диагностический grid search по параметрам кластеризации.
# Это не обучение модели, а способ понять чувствительность baseline к порогам, и выбрать оптимальные параметры для дальнейшей работы
RUN_CLUSTERING_GRID_SEARCH = False
GRID_STORY_THRESHOLDS = [0.76, 0.78, 0.80, 0.82, 0.84, 0.86]
GRID_STORY_WINDOW_DAYS = [3, 7, 14]

RUN_NOVELTY_GRID_SEARCH = False

GRID_MINOR_THRESHOLDS = [0.78, 0.80, 0.82, 0.84, 0.86, 0.88]
GRID_DUPLICATE_THRESHOLDS = [0.90, 0.92, 0.94, 0.96]

RANDOM_STATE = 42


## 3. Схемы и вспомогательные функции


In [40]:
CLEAN_LIKE_COLUMNS = [
    "news_id",
    "url",
    "title",
    "text",
    "topic",
    "tags",
    "published_at",
    "text_length",
    "text_num_words",
    "title_length",
    "title_num_words",
    "model_text",
    "model_length",
    "model_num_words",
]

SIMPLE_ANNOTATION_COLUMNS = [
    "news_id",
    "published_at",
    "topic",
    "title",
    "text",
    "cluster_id",
    "novelty_label",
    "comment",
    "needs_review",
]

PREDICTION_COLUMNS = SIMPLE_ANNOTATION_COLUMNS.copy()

VALID_NOVELTY_LABELS = ["significant", "minor", "duplicate", "wrong_cluster", "unclear"]
IMPORTANT_POSITIVE_LABEL = "significant"
IMPORTANT_NEGATIVE_LABELS = ["minor", "duplicate"]
IMPORTANT_EXCLUDED_LABELS = ["", "wrong_cluster", "unclear"]


def count_words(value: str) -> int:
    value = "" if pd.isna(value) else str(value)
    return len(re.findall(r"\w+", value, flags=re.UNICODE))


def normalize_news_id(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_bool_like(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    value = str(value).strip().lower()
    return value in {"1", "true", "yes", "y", "да", "истина"}


def dataframe_content_hash(df: pd.DataFrame, columns: list[str]) -> str:
    h = hashlib.sha256()
    safe = df[columns].copy()
    for col in columns:
        safe[col] = safe[col].fillna("").astype(str)
    csv_bytes = safe.to_csv(index=False).encode("utf-8")
    h.update(csv_bytes)
    return h.hexdigest()[:16]


## 4. Подготовка входных данных

Baseline принимает clean-like формат, но для удобства может читать файлы в новой простой схеме разметки и приводить их к clean-like виду.

Новая простая схема должна содержать минимум:

```text
news_id, published_at, topic, title, text
```

Если в файле уже есть `cluster_id`, `novelty_label`, `comment`, `needs_review`, baseline их игнорирует. Эти поля будут заново сформированы на выходе.


In [41]:
def prepare_model_input(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Привести входной файл к clean-like формату для baseline.

    Поддерживаются два типа входа:
    - clean-like датасет, аналогичный `lenta_clean_news.csv`;
    - файл в новой простой схеме разметки/темплейта:
      news_id, published_at, topic, title, text, cluster_id, novelty_label, comment, needs_review.

    Лишние колонки игнорируются.
    Недостающие технические колонки пересчитываются.
    """
    df = raw_df.copy()

    required_base = ["news_id", "published_at", "topic", "title", "text"]
    missing = [col for col in required_base if col not in df.columns]
    if missing:
        raise ValueError(f"Во входном файле отсутствуют обязательные колонки: {missing}")

    df["news_id"] = normalize_news_id(df["news_id"])
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    df["topic"] = df["topic"].fillna("unknown").astype(str)

    if df["published_at"].isna().any():
        bad_count = int(df["published_at"].isna().sum())
        raise ValueError(f"Найдено строк с некорректным published_at: {bad_count}")

    if "url" not in df.columns:
        df["url"] = ""
    if "tags" not in df.columns:
        df["tags"] = ""

    if "model_text" not in df.columns:
        df["model_text"] = (
            df["title"].fillna("")
            + "\n\n"
            + df["text"].fillna("")
        ).str.strip()
    else:
        df["model_text"] = df["model_text"].fillna("").astype(str)
        empty_model_text_mask = df["model_text"].str.strip().eq("")
        df.loc[empty_model_text_mask, "model_text"] = (
            df.loc[empty_model_text_mask, "title"].fillna("")
            + "\n\n"
            + df.loc[empty_model_text_mask, "text"].fillna("")
        ).str.strip()

    # Пересчитываем length-колонки, чтобы annotation-like вход тоже стал clean-like.
    df["text_length"] = df["text"].str.len()
    df["text_num_words"] = df["text"].map(count_words)
    df["title_length"] = df["title"].str.len()
    df["title_num_words"] = df["title"].map(count_words)
    df["model_length"] = df["model_text"].str.len()
    df["model_num_words"] = df["model_text"].map(count_words)

    df = df.sort_values(["published_at", "news_id"]).reset_index(drop=True)
    return df[CLEAN_LIKE_COLUMNS]


def prediction_output_from_model_input(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["news_id", "published_at", "topic", "title", "text"]].copy()
    out["cluster_id"] = ""
    out["novelty_label"] = ""
    out["comment"] = ""
    out["needs_review"] = False
    return out[PREDICTION_COLUMNS]


## 5. Загрузка входного файла


In [42]:
raw_input = pd.read_csv(INPUT_PATH)
model_input = prepare_model_input(raw_input)

print("Входной файл:", INPUT_PATH)
print("Размер исходного файла:", raw_input.shape)
print("Размер clean-like входа модели:", model_input.shape)
print("Период:", model_input["published_at"].min(), "—", model_input["published_at"].max())
print("Распределение по темам:")
print(model_input["topic"].value_counts())

model_input.head(3)


Входной файл: E:\ML\Projects\Git\news-flow-analysis\data\prepared\lenta_golden_candidate_pool.csv
Размер исходного файла: (3176, 16)
Размер clean-like входа модели: (3176, 14)
Период: 2004-03-01 00:00:00 — 2004-04-30 00:00:00
Распределение по темам:
topic
Россия       1498
Мир          1370
Экономика     308
Name: count, dtype: int64


,news_id,url,title,text,topic,tags,published_at,text_length,text_num_words,title_length,title_num_words,model_text,model_length,model_num_words
0,88522,https://lenta.ru/news/2004/03/01/nato/,Расширение НАТО случится досрочно,Церемония принятия в НАТО новых членов состоит...,Мир,Все,2004-03-01,762,106,33,4,Расширение НАТО случится досрочно. Церемония п...,797,110
1,88525,https://lenta.ru/news/2004/03/01/kaloev/,Подозреваемый в убийстве диспетчера Skyguide г...,"Немецкие адвокаты, которые представляют интере...",Мир,Все,2004-03-01,2623,343,72,9,Подозреваемый в убийстве диспетчера Skyguide г...,2697,352
2,88528,https://lenta.ru/news/2004/03/01/constitution/,Правительственный совет Ирака одобрил проект в...,Члены Правительственного совета Ирака приняли ...,Мир,Все,2004-03-01,1371,168,66,7,Правительственный совет Ирака одобрил проект в...,1439,175


## 6. Реализация baseline

Baseline собран в переиспользуемый класс, чтобы ту же логику потом можно было перенести из ноутбука в сервис.

Класс не обучает supervised-модель. Метод `fit()` только готовит переиспользуемое состояние:

- загружает или скачивает pretrained SentenceTransformer;
- при необходимости сохраняет модель локально;
- считает или загружает из кеша embeddings для текущего входного набора.

Изменение порогов не требует заново сохранять SentenceTransformer и не требует пересчитывать embeddings. Нужно только заново выполнить кластеризацию и novelty-разметку.


In [43]:
@dataclass
class SemanticNewsBaselineConfig:
    model_name: str
    local_model_dir: Path
    embeddings_cache_dir: Path
    force_redownload_model: bool = False
    force_recompute_embeddings: bool = False
    story_threshold: float = 0.82
    story_window_days: int = 7
    minor_threshold: float = 0.78
    duplicate_threshold: float = 0.92
    review_margin: float = 0.03
    use_topic_blocking: bool = True
    batch_size: int = 32


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


class SemanticNewsBaseline:
    """Семантический baseline для кластеризации новостных сюжетов и novelty-разметки.

    Вход:
        clean-like dataframe с news_id, title, text, topic, published_at, model_text.

    Выход:
        prediction-like dataframe:
        news_id, published_at, topic, title, text, cluster_id, novelty_label, comment, needs_review.

    Класс намеренно устроен просто:
    - готовые sentence-transformer embeddings;
    - граф похожести внутри topic с временным окном;
    - компоненты связности как сюжетные кластеры;
    - rule-based novelty-метки по максимальной близости к предыдущим новостям кластера.
    """

    def __init__(self, config: SemanticNewsBaselineConfig):
        self.config = config
        self.model = None
        self.df_: Optional[pd.DataFrame] = None
        self.embeddings_: Optional[np.ndarray] = None
        self.last_graph_edges_: Optional[int] = None
        self.last_cluster_count_: Optional[int] = None

    def fit(self, df: pd.DataFrame) -> "SemanticNewsBaseline":
        self.df_ = prepare_model_input(df)
        self.model = self._load_sentence_transformer_model()
        self.embeddings_ = self._compute_or_load_embeddings(self.df_)
        return self

    def predict(
        self,
        story_threshold: Optional[float] = None,
        story_window_days: Optional[int] = None,
        minor_threshold: Optional[float] = None,
        duplicate_threshold: Optional[float] = None,
        review_margin: Optional[float] = None,
        use_topic_blocking: Optional[bool] = None,
    ) -> pd.DataFrame:
        if self.df_ is None or self.embeddings_ is None:
            raise RuntimeError("Сначала вызовите fit(df), затем predict().")

        story_threshold = self.config.story_threshold if story_threshold is None else story_threshold
        story_window_days = self.config.story_window_days if story_window_days is None else story_window_days
        minor_threshold = self.config.minor_threshold if minor_threshold is None else minor_threshold
        duplicate_threshold = self.config.duplicate_threshold if duplicate_threshold is None else duplicate_threshold
        review_margin = self.config.review_margin if review_margin is None else review_margin
        use_topic_blocking = self.config.use_topic_blocking if use_topic_blocking is None else use_topic_blocking

        working_df = self.df_.copy().reset_index(drop=True)
        cluster_ids = self._build_semantic_graph_clusters(
            working_df,
            self.embeddings_,
            story_threshold=story_threshold,
            story_window_days=story_window_days,
            use_topic_blocking=use_topic_blocking,
        )
        working_df["baseline_cluster_id"] = cluster_ids.values

        predictions = self._label_novelty_within_clusters(
            working_df,
            self.embeddings_,
            cluster_col="baseline_cluster_id",
            minor_threshold=minor_threshold,
            duplicate_threshold=duplicate_threshold,
            review_margin=review_margin,
        )
        return predictions

    def fit_predict(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).predict()

    def save_config(self, path: Path) -> None:
        payload = asdict(self.config)
        payload = {key: str(value) if isinstance(value, Path) else value for key, value in payload.items()}
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def _load_sentence_transformer_model(self):
        from sentence_transformers import SentenceTransformer

        self.config.local_model_dir.mkdir(parents=True, exist_ok=True)

        if self.config.local_model_dir.exists() and any(self.config.local_model_dir.iterdir()) and not self.config.force_redownload_model:
            print(f"Загружаю локальную sentence-transformer модель из {self.config.local_model_dir}")
            return SentenceTransformer(str(self.config.local_model_dir))

        print(f"Загружаю sentence-transformer модель из Hugging Face: {self.config.model_name}")
        model = SentenceTransformer(self.config.model_name)
        model.save(str(self.config.local_model_dir))
        print(f"Локальная копия sentence-transformer сохранена в {self.config.local_model_dir}")
        return model

    def _get_embeddings_cache_path(self, df: pd.DataFrame) -> Path:
        input_hash = dataframe_content_hash(df, ["news_id", "published_at", "model_text"])
        model_hash = hashlib.sha256(self.config.model_name.encode("utf-8")).hexdigest()[:8]
        return self.config.embeddings_cache_dir / f"embeddings_{model_hash}_{input_hash}.npz"

    def _compute_or_load_embeddings(self, df: pd.DataFrame) -> np.ndarray:
        self.config.embeddings_cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = self._get_embeddings_cache_path(df)

        if cache_path.exists() and not self.config.force_recompute_embeddings:
            print(f"Загружаю embeddings из кеша: {cache_path}")
            data = np.load(cache_path, allow_pickle=False)
            embeddings = data["embeddings"]
            if embeddings.shape[0] != len(df):
                raise ValueError("Количество строк в кеше embeddings не совпадает с входным dataframe")
            return embeddings

        print("Считаю embeddings...")
        embeddings = self.model.encode(
            df["model_text"].tolist(),
            batch_size=self.config.batch_size,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        np.savez_compressed(cache_path, embeddings=embeddings)
        print(f"Embeddings сохранены в кеш: {cache_path}")
        return embeddings

    def _build_semantic_graph_clusters(
        self,
        df: pd.DataFrame,
        embeddings: np.ndarray,
        story_threshold: float,
        story_window_days: int,
        use_topic_blocking: bool = True,
    ) -> pd.Series:
        n = len(df)
        uf = UnionFind(n)
        time_values = df["published_at"].to_numpy()

        if use_topic_blocking:
            groups = df.groupby("topic", sort=False).indices.values()
        else:
            groups = [np.arange(n)]

        max_delta = np.timedelta64(story_window_days, "D")
        total_edges = 0

        for raw_indices in groups:
            indices = np.array(list(raw_indices), dtype=int)
            if len(indices) <= 1:
                continue

            indices = indices[np.argsort(time_values[indices])]

            for local_i, global_i in enumerate(indices):
                j = local_i + 1
                while j < len(indices):
                    global_j = indices[j]
                    if time_values[global_j] - time_values[global_i] > max_delta:
                        break
                    sim = float(np.dot(embeddings[global_i], embeddings[global_j]))
                    if sim >= story_threshold:
                        uf.union(global_i, global_j)
                        total_edges += 1
                    j += 1

        roots = [uf.find(i) for i in range(n)]
        root_to_cluster = {}
        cluster_ids = []
        for root in roots:
            if root not in root_to_cluster:
                root_to_cluster[root] = f"baseline_cluster_{len(root_to_cluster):05d}"
            cluster_ids.append(root_to_cluster[root])

        self.last_graph_edges_ = total_edges
        self.last_cluster_count_ = len(set(cluster_ids))

        print("Рёбер в графе похожести:", total_edges)
        print("Количество кластеров:", self.last_cluster_count_)
        return pd.Series(cluster_ids, index=df.index)

    def _label_novelty_within_clusters(
        self,
        df: pd.DataFrame,
        embeddings: np.ndarray,
        cluster_col: str,
        minor_threshold: float,
        duplicate_threshold: float,
        review_margin: float,
    ) -> pd.DataFrame:
        result = prediction_output_from_model_input(df)
        result["cluster_id"] = df[cluster_col].values

        labels = [""] * len(df)
        comments = [""] * len(df)
        needs_review = [False] * len(df)

        for cluster_id, cluster_indices_raw in df.groupby(cluster_col, sort=False).indices.items():
            cluster_indices = list(cluster_indices_raw)
            cluster_indices = sorted(
                cluster_indices,
                key=lambda idx: (df.loc[idx, "published_at"], df.loc[idx, "news_id"]),
            )

            if not cluster_indices:
                continue

            first_idx = cluster_indices[0]
            comments[first_idx] = "старт_сюжета"
            labels[first_idx] = ""
            needs_review[first_idx] = False

            previous_indices = [first_idx]
            for idx in cluster_indices[1:]:
                sims = embeddings[previous_indices] @ embeddings[idx]
                best_pos = int(np.argmax(sims))
                max_sim = float(sims[best_pos])
                matched_idx = previous_indices[best_pos]
                matched_news_id = df.loc[matched_idx, "news_id"]

                if max_sim >= duplicate_threshold:
                    label = "duplicate"
                elif max_sim >= minor_threshold:
                    label = "minor"
                else:
                    label = "significant"

                near_threshold = (
                    abs(max_sim - duplicate_threshold) <= review_margin
                    or abs(max_sim - minor_threshold) <= review_margin
                )

                labels[idx] = label
                comments[idx] = (
                    f"max_prev_sim={max_sim:.3f}; "
                    f"matched_news_id={matched_news_id}; "
                    f"minor_threshold={minor_threshold:.2f}; "
                    f"duplicate_threshold={duplicate_threshold:.2f}"
                )
                needs_review[idx] = bool(near_threshold)
                previous_indices.append(idx)

        result["novelty_label"] = labels
        result["comment"] = comments
        result["needs_review"] = needs_review
        return result[PREDICTION_COLUMNS]


## 7. Запуск baseline


In [44]:
baseline_config = SemanticNewsBaselineConfig(
    model_name=MODEL_NAME,
    local_model_dir=LOCAL_MODEL_DIR,
    embeddings_cache_dir=EMBEDDINGS_CACHE_DIR,
    force_redownload_model=FORCE_REDOWNLOAD_MODEL,
    force_recompute_embeddings=FORCE_RECOMPUTE_EMBEDDINGS,
    story_threshold=STORY_THRESHOLD,
    story_window_days=STORY_WINDOW_DAYS,
    minor_threshold=MINOR_THRESHOLD,
    duplicate_threshold=DUPLICATE_THRESHOLD,
    review_margin=REVIEW_MARGIN,
    use_topic_blocking=USE_TOPIC_BLOCKING,
)

baseline = SemanticNewsBaseline(baseline_config)
baseline.fit(model_input)
baseline.save_config(BASELINE_ARTIFACTS_DIR / "semantic_baseline_config.json")

if OUTPUT_PATH.exists() and not FORCE_RERUN_BASELINE:
    print(f"Загружаю существующий файл predictions baseline: {OUTPUT_PATH}")
    baseline_predictions = pd.read_csv(OUTPUT_PATH)
else:
    baseline_predictions = baseline.predict()
    baseline_predictions.to_csv(OUTPUT_PATH, index=False)
    print("Predictions baseline сохранены:", OUTPUT_PATH)

print("Размер predictions:", baseline_predictions.shape)
print("Количество кластеров:", baseline_predictions["cluster_id"].nunique())
print("Распределение novelty_label:")
print(baseline_predictions["novelty_label"].replace("", "<старт_сюжета>").value_counts())
print("Доля needs_review:", baseline_predictions["needs_review"].map(normalize_bool_like).mean())

baseline_predictions.head(5)


Загружаю локальную sentence-transformer модель из E:\ML\Projects\Git\news-flow-analysis\data\artifacts\baseline\sentence_transformer_bge_m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Загружаю embeddings из кеша: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\baseline\embeddings_cache\embeddings_d790e737_3cb8b68e5c8157fa.npz
Рёбер в графе похожести: 914
Количество кластеров: 2607
Predictions baseline сохранены: E:\ML\Projects\Git\news-flow-analysis\data\prepared\lenta_baseline_predictions.csv
Размер predictions: (3176, 9)
Количество кластеров: 2607
Распределение novelty_label:
novelty_label
<старт_сюжета>    2607
significant        427
minor               72
duplicate           70
Name: count, dtype: int64
Доля needs_review: 0.09319899244332494


,news_id,published_at,topic,title,text,cluster_id,novelty_label,comment,needs_review
0,88522,2004-03-01,Мир,Расширение НАТО случится досрочно,Церемония принятия в НАТО новых членов состоит...,baseline_cluster_00000,,старт_сюжета,False
1,88525,2004-03-01,Мир,Подозреваемый в убийстве диспетчера Skyguide г...,"Немецкие адвокаты, которые представляют интере...",baseline_cluster_00001,,старт_сюжета,False
2,88528,2004-03-01,Мир,Правительственный совет Ирака одобрил проект в...,Члены Правительственного совета Ирака приняли ...,baseline_cluster_00002,,старт_сюжета,False
3,88534,2004-03-01,Мир,Совет безопасности ООН одобрил отправку миротв...,Члены Совета безопасности ООН в воскресенье ве...,baseline_cluster_00003,,старт_сюжета,False
4,88536,2004-03-01,Россия,Дагестанская милиция нашла в горах погибших по...,В Цунтинском районе Дагестана обнаружены следы...,baseline_cluster_00004,,старт_сюжета,False


## 8. Вспомогательные функции для inline evaluation

Этот блок нужен, чтобы быстро оценить baseline на human gold прямо в `05`, не переключаясь в `04`.

Основной evaluation notebook всё равно остаётся `04`, но для подбора и диагностики baseline удобно видеть основные метрики здесь.

Метрики разделены на две группы:

1. **Основные** — те, которые напрямую отражают качество задачи и продуктовые риски.
2. **Второстепенные** — диагностические показатели, полезные для анализа, но не являющиеся главными критериями.


In [45]:
def load_prediction_like(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = SIMPLE_ANNOTATION_COLUMNS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"В файле {path} отсутствуют колонки: {missing}")

    df = df[required].copy()
    df["news_id"] = normalize_news_id(df["news_id"])
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df["cluster_id"] = df["cluster_id"].fillna("").astype(str).str.strip()
    df["novelty_label"] = df["novelty_label"].fillna("").astype(str).str.strip()
    df["needs_review"] = df["needs_review"].map(normalize_bool_like)
    return df


def compute_pairwise_clustering_metrics(reference_cluster_ids: Iterable[str], candidate_cluster_ids: Iterable[str]) -> dict:
    ref = np.array(list(reference_cluster_ids), dtype=object)
    cand = np.array(list(candidate_cluster_ids), dtype=object)
    n = len(ref)

    tp = fp = fn = tn = 0
    for i in range(n):
        for j in range(i + 1, n):
            ref_same = ref[i] == ref[j]
            cand_same = cand[i] == cand[j]
            if ref_same and cand_same:
                tp += 1
            elif not ref_same and cand_same:
                fp += 1
            elif ref_same and not cand_same:
                fn += 1
            else:
                tn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    false_merge_rate = fp / (tp + fp) if (tp + fp) else 0.0
    false_split_rate = fn / (tp + fn) if (tp + fn) else 0.0

    return {
        "n_items": n,
        "total_pairs": n * (n - 1) // 2,
        "tp_same_pairs": tp,
        "fp_false_merge_pairs": fp,
        "fn_false_split_pairs": fn,
        "tn_different_pairs": tn,
        "reference_same_pairs": tp + fn,
        "candidate_same_pairs": tp + fp,
        "pairwise_precision": precision,
        "pairwise_recall": recall,
        "pairwise_f1": f1,
        "false_merge_rate": false_merge_rate,
        "false_split_rate": false_split_rate,
        "adjusted_rand_index": adjusted_rand_score(ref, cand),
        "normalized_mutual_info": normalized_mutual_info_score(ref, cand),
    }


def to_important_binary(label: str):
    label = "" if pd.isna(label) else str(label).strip()
    if label == IMPORTANT_POSITIVE_LABEL:
        return 1
    if label in IMPORTANT_NEGATIVE_LABELS:
        return 0
    return np.nan


def evaluate_predictions(reference: pd.DataFrame, candidate: pd.DataFrame) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    merged = reference.merge(candidate, on="news_id", suffixes=("_ref", "_cand"), how="inner")

    coverage = len(merged) / len(reference) if len(reference) else 0.0

    clustering_metrics = compute_pairwise_clustering_metrics(
        merged["cluster_id_ref"],
        merged["cluster_id_cand"],
    )

    # Multiclass novelty metrics: только строки, где обе стороны выдали валидную непустую метку.
    ref_label = merged["novelty_label_ref"].fillna("").astype(str).str.strip()
    cand_label = merged["novelty_label_cand"].fillna("").astype(str).str.strip()
    valid_multiclass_mask = ref_label.isin(VALID_NOVELTY_LABELS) & cand_label.isin(VALID_NOVELTY_LABELS)

    novelty_metrics = {
        "novelty_eval_rows": int(valid_multiclass_mask.sum()),
        "novelty_accuracy": np.nan,
        "novelty_macro_f1": np.nan,
        "novelty_weighted_f1": np.nan,
        "cohen_kappa": np.nan,
    }

    if valid_multiclass_mask.any():
        y_ref = ref_label[valid_multiclass_mask]
        y_cand = cand_label[valid_multiclass_mask]
        novelty_metrics.update({
            "novelty_accuracy": accuracy_score(y_ref, y_cand),
            "novelty_macro_f1": f1_score(y_ref, y_cand, labels=VALID_NOVELTY_LABELS, average="macro", zero_division=0),
            "novelty_weighted_f1": f1_score(y_ref, y_cand, labels=VALID_NOVELTY_LABELS, average="weighted", zero_division=0),
            "cohen_kappa": cohen_kappa_score(y_ref, y_cand),
        })

    # Important-update metrics: significant против minor/duplicate.
    ref_binary = ref_label.map(to_important_binary)
    cand_binary = cand_label.map(to_important_binary)
    valid_binary_mask = ref_binary.notna() & cand_binary.notna()

    important_metrics = {
        "important_eval_rows": int(valid_binary_mask.sum()),
        "important_precision": np.nan,
        "important_recall": np.nan,
        "important_f1": np.nan,
    }

    if valid_binary_mask.any():
        y_ref_bin = ref_binary[valid_binary_mask].astype(int)
        y_cand_bin = cand_binary[valid_binary_mask].astype(int)
        important_metrics.update({
            "important_precision": precision_score(y_ref_bin, y_cand_bin, zero_division=0),
            "important_recall": recall_score(y_ref_bin, y_cand_bin, zero_division=0),
            "important_f1": f1_score(y_ref_bin, y_cand_bin, zero_division=0),
        })

    summary = {
        "reference_rows": len(reference),
        "candidate_rows": len(candidate),
        "overlap_rows": len(merged),
        "coverage": coverage,
        **clustering_metrics,
        **novelty_metrics,
        **important_metrics,
    }

    summary_df = pd.DataFrame.from_dict(summary, orient="index", columns=["value"])
    return merged, summary, summary_df


def metric_status(value, target, direction: str) -> str:
    if pd.isna(value) or target is None:
        return "INFO"
    if direction == ">=":
        return "OK" if value >= target else "WARN"
    if direction == "<=":
        return "OK" if value <= target else "WARN"
    return "INFO"


MAIN_METRIC_SPECS = [
    {
        "metric": "coverage",
        "name_ru": "Покрытие reference по news_id",
        "target": 0.95,
        "direction": ">=",
        "comment": "Sanity-check: сравниваются ли одни и те же новости.",
    },
    {
        "metric": "pairwise_f1",
        "name_ru": "Pairwise F1 кластеризации",
        "target": 0.75,
        "direction": ">=",
        "comment": "Основная метрика качества группировки инфоповодов.",
    },
    {
        "metric": "false_merge_rate",
        "name_ru": "Доля ложных склеек",
        "target": 0.15,
        "direction": "<=",
        "comment": "Критичный риск: разные сюжеты ошибочно объединяются.",
    },
    {
        "metric": "important_recall",
        "name_ru": "Recall важных обновлений",
        "target": 0.80,
        "direction": ">=",
        "comment": "Критичный риск: модель не должна пропускать significant updates.",
    },
    {
        "metric": "important_f1",
        "name_ru": "F1 важных обновлений",
        "target": 0.75,
        "direction": ">=",
        "comment": "Баланс precision/recall по классу significant.",
    },
]


SECONDARY_METRICS = [
    "pairwise_precision",
    "pairwise_recall",
    "false_split_rate",
    "adjusted_rand_index",
    "normalized_mutual_info",
    "novelty_eval_rows",
    "novelty_accuracy",
    "novelty_macro_f1",
    "novelty_weighted_f1",
    "cohen_kappa",
    "important_eval_rows",
    "important_precision",
    "n_items",
    "total_pairs",
    "tp_same_pairs",
    "fp_false_merge_pairs",
    "fn_false_split_pairs",
    "tn_different_pairs",
    "reference_same_pairs",
    "candidate_same_pairs",
]


def split_evaluation_summary(summary: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    main_rows = []
    for spec in MAIN_METRIC_SPECS:
        value = summary.get(spec["metric"], np.nan)
        main_rows.append({
            "metric": spec["metric"],
            "name_ru": spec["name_ru"],
            "value": value,
            "target": f'{spec["direction"]} {spec["target"]}',
            "status": metric_status(value, spec["target"], spec["direction"]),
            "comment": spec["comment"],
        })

    secondary_rows = []
    for metric in SECONDARY_METRICS:
        if metric in summary:
            secondary_rows.append({
                "metric": metric,
                "value": summary.get(metric),
            })

    return pd.DataFrame(main_rows), pd.DataFrame(secondary_rows)


## 9. Inline evaluation относительно reference

Если `REFERENCE_PATH` существует, baseline сразу сравнивается с reference-файлом. Обычно здесь указывается `lenta_golden_set_human.csv`.

Важно: если `coverage` низкий, метрики нельзя интерпретировать. Это значит, что baseline predictions и reference сделаны на разных поколениях файлов или разных наборах `news_id`.


In [46]:
if RUN_INLINE_EVALUATION and REFERENCE_PATH.exists():
    reference_df = load_prediction_like(REFERENCE_PATH)
    candidate_df = load_prediction_like(OUTPUT_PATH)
    merged_eval, eval_summary, eval_summary_df = evaluate_predictions(reference_df, candidate_df)
    main_metrics_df, secondary_metrics_df = split_evaluation_summary(eval_summary)

    print("Основные характеристики сравнения:")
    display(main_metrics_df)

    print("Второстепенные диагностические метрики:")
    display(secondary_metrics_df)

    if eval_summary["coverage"] < 0.95:
        print(
            "ВНИМАНИЕ: низкое покрытие reference по news_id. "
            "Скорее всего, reference и baseline predictions относятся к разным версиям данных."
        )

    reports_dir = ARTIFACTS_DIR / "inline_evaluation_reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    merged_eval.to_csv(reports_dir / "baseline_inline_merged_comparison.csv", index=False)
    eval_summary_df.to_csv(reports_dir / "baseline_inline_summary_all_metrics.csv")
    main_metrics_df.to_csv(reports_dir / "baseline_inline_summary_main_metrics.csv", index=False)
    secondary_metrics_df.to_csv(reports_dir / "baseline_inline_summary_secondary_metrics.csv", index=False)
    print("Отчёты inline evaluation сохранены в:", reports_dir)
else:
    print("Inline evaluation пропущен. Укажите RUN_INLINE_EVALUATION=True и существующий REFERENCE_PATH.")


Основные характеристики сравнения:


,metric,name_ru,value,target,status,comment
0,coverage,Покрытие reference по news_id,1.000000,>= 0.95,OK,Sanity-check: сравниваются ли одни и те же нов...
1,pairwise_f1,Pairwise F1 кластеризации,0.769737,>= 0.75,OK,Основная метрика качества группировки инфопово...
2,false_merge_rate,Доля ложных склеек,0.000000,<= 0.15,OK,Критичный риск: разные сюжеты ошибочно объедин...
3,important_recall,Recall важных обновлений,0.725490,>= 0.8,WARN,Критичный риск: модель не должна пропускать si...
4,important_f1,F1 важных обновлений,0.770833,>= 0.75,OK,Баланс precision/recall по классу significant.


Второстепенные диагностические метрики:


,metric,value
0,pairwise_precision,1.000000
1,pairwise_recall,0.625668
2,false_split_rate,0.374332
3,adjusted_rand_index,0.765079
4,normalized_mutual_info,0.941666
5,novelty_eval_rows,63.000000
6,novelty_accuracy,0.634921
7,novelty_macro_f1,0.232738
8,novelty_weighted_f1,0.659722
9,cohen_kappa,0.077070


Отчёты inline evaluation сохранены в: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\inline_evaluation_reports


## 10. Диагностический grid search по параметрам кластеризации

Этот блок нужен, чтобы понять жёсткость проблемы и чувствительность baseline к порогам.

Важно: если использовать human gold для выбора параметров, это уже будет ручная калибровка. Для учебной baseline-версии можно показать grid как диагностику, но финальные параметры лучше зафиксировать заранее или явно описать как подобранные на dev-subset.


In [47]:
def run_grid_search_if_enabled():
    if not RUN_CLUSTERING_GRID_SEARCH:
        print("Grid search пропущен. Чтобы запустить, установите RUN_CLUSTERING_GRID_SEARCH=True.")
        return None
    if not REFERENCE_PATH.exists():
        print("Grid search пропущен: reference-файл не найден.")
        return None

    reference_df = load_prediction_like(REFERENCE_PATH)
    rows = []

    for story_window_days in GRID_STORY_WINDOW_DAYS:
        for story_threshold in GRID_STORY_THRESHOLDS:
            preds = baseline.predict(
                story_threshold=story_threshold,
                story_window_days=story_window_days,
                minor_threshold=MINOR_THRESHOLD,
                duplicate_threshold=DUPLICATE_THRESHOLD,
                review_margin=REVIEW_MARGIN,
                use_topic_blocking=USE_TOPIC_BLOCKING,
            )
            _, summary, _ = evaluate_predictions(reference_df, preds)
            rows.append({
                "story_threshold": story_threshold,
                "story_window_days": story_window_days,
                "minor_threshold": MINOR_THRESHOLD,
                "duplicate_threshold": DUPLICATE_THRESHOLD,
                "coverage": summary["coverage"],
                "pairwise_precision": summary["pairwise_precision"],
                "pairwise_recall": summary["pairwise_recall"],
                "pairwise_f1": summary["pairwise_f1"],
                "false_merge_rate": summary["false_merge_rate"],
                "false_split_rate": summary["false_split_rate"],
                "important_precision": summary["important_precision"],
                "important_recall": summary["important_recall"],
                "important_f1": summary["important_f1"],
                "novelty_macro_f1": summary["novelty_macro_f1"],
            })

    grid_df = pd.DataFrame(rows)
    grid_path = ARTIFACTS_DIR / "baseline_grid_search_results.csv"
    grid_df.to_csv(grid_path, index=False)
    print("Результаты grid search сохранены:", grid_path)
    return grid_df.sort_values(["false_merge_rate", "pairwise_f1"], ascending=[True, False])


grid_results = run_grid_search_if_enabled()
if grid_results is not None:
    display(grid_results.head(20))


Grid search пропущен. Чтобы запустить, установите RUN_CLUSTERING_GRID_SEARCH=True.


In [48]:
novelty_grid_rows = []

if RUN_NOVELTY_GRID_SEARCH:
    for minor_threshold in GRID_MINOR_THRESHOLDS:
        for duplicate_threshold in GRID_DUPLICATE_THRESHOLDS:
            if duplicate_threshold <= minor_threshold:
                continue

            config = SemanticNewsBaselineConfig(
                model_name=MODEL_NAME,
                local_model_dir=LOCAL_MODEL_DIR,
                embeddings_cache_dir=EMBEDDINGS_CACHE_DIR,
                story_threshold=STORY_THRESHOLD,
                story_window_days=STORY_WINDOW_DAYS,
                minor_threshold=minor_threshold,
                duplicate_threshold=duplicate_threshold,
                review_margin=REVIEW_MARGIN,
                force_redownload_model=False,
                force_recompute_embeddings=False,
            )
           

            baseline = SemanticNewsBaseline(config)
            predictions = baseline.fit_predict(model_input)

            _, summary, _ = evaluate_predictions(
                reference=reference_df,
                candidate=predictions,
            )

            row = {
                "minor_threshold": minor_threshold,
                "duplicate_threshold": duplicate_threshold,
                **summary,
            }

            novelty_grid_rows.append(row)

    novelty_grid_results = pd.DataFrame(novelty_grid_rows)

    novelty_grid_results.to_csv(
        ARTIFACTS_DIR / "baseline_novelty_grid_search_results.csv",
        index=False,
    )

    display(
        novelty_grid_results.sort_values(
            ["important_f1", "important_recall", "important_precision"],
            ascending=[False, False, False],
        ).head(20)
    )

## 11. Сохранение baseline summary


In [49]:
summary = {
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "model_name": MODEL_NAME,
    "local_model_dir": str(LOCAL_MODEL_DIR),
    "baseline_config_path": str(BASELINE_ARTIFACTS_DIR / "semantic_baseline_config.json"),
    "n_rows": int(len(model_input)),
    "period_min": str(model_input["published_at"].min()),
    "period_max": str(model_input["published_at"].max()),
    "story_threshold": STORY_THRESHOLD,
    "story_window_days": STORY_WINDOW_DAYS,
    "minor_threshold": MINOR_THRESHOLD,
    "duplicate_threshold": DUPLICATE_THRESHOLD,
    "review_margin": REVIEW_MARGIN,
    "use_topic_blocking": USE_TOPIC_BLOCKING,
    "graph_edges": int(baseline.last_graph_edges_) if baseline.last_graph_edges_ is not None else None,
    "n_clusters": int(baseline_predictions["cluster_id"].nunique()),
    "label_distribution": baseline_predictions["novelty_label"].replace("", "<старт_сюжета>").value_counts().to_dict(),
}

if RUN_INLINE_EVALUATION and REFERENCE_PATH.exists():
    summary["inline_evaluation_reference_path"] = str(REFERENCE_PATH)
    summary["inline_evaluation_all_metrics"] = eval_summary
    summary["inline_evaluation_main_metrics"] = main_metrics_df.to_dict(orient="records")
    summary["inline_evaluation_secondary_metrics"] = secondary_metrics_df.to_dict(orient="records")

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Baseline summary сохранён:", SUMMARY_PATH)


Baseline summary сохранён: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\baseline\lenta_baseline_summary.json


## 12. Заметки по бейзлайну

В этом ноутбуке baseline вынесен в переиспользуемый класс `SemanticNewsBaseline`.

Практические моменты:

- Изменение `STORY_THRESHOLD`, `STORY_WINDOW_DAYS`, `MINOR_THRESHOLD`, `DUPLICATE_THRESHOLD` или `REVIEW_MARGIN` не требует заново скачивать или сохранять SentenceTransformer.
- Эти изменения также не требуют пересчитывать embeddings, потому что embeddings зависят от входных текстов и embedding-модели, а не от порогов кластеризации.
- Чтобы принудительно пересчитать embeddings, установите `FORCE_RECOMPUTE_EMBEDDINGS = True`.
- Чтобы принудительно заново скачать/сохранить модель, установите `FORCE_REDOWNLOAD_MODEL = True`.
- Чтобы пересобрать prediction CSV после изменения параметров, установите `FORCE_RERUN_BASELINE = True`.

Финальная конфигурация baseline выбрана по результатам небольшого grid search. 
Для кластеризации выбран консервативный режим: высокий порог семантической близости и временное окно 14 дней. 
Это позволило сохранить низкий уровень ложных склеек сюжетов.

Для novelty-разметки порог `MINOR_THRESHOLD` был повышен до 0.88, так как исходная конфигурация слишком редко помечала новости как `significant` и давала низкий recall важных обновлений.

Baseline достигает приемлемого качества кластеризации (`pairwise_f1 ≈ 0.77` при целевых 0.75-0.8) при отсутствии ложных склеек (`false_merge_rate = 0.0`). 
Для важных обновлений baseline также показывает рабочий уровень (`important_f1 ≈ 0.77` при целевом 0.75), но recall всё ещё ниже целевого (`important_recall ≈ 0.73` при целевом 0.8).

Это делает baseline полезной точкой отсчёта: он уже не является совсем бесполезной моделью, но оставляет пространство для улучшения в основной модели.

## 13. Делаем полноценную модель

Дорабатываем бейзлайн, пытаясь довести показатели до целевых.
Основная идея - усложнить кластеризацию, увеличив F1, но при этом чтобы false_merge_rate не превысил порога (0.15). Так как кластирация в целом и на бейзлайне неплохая - сильно не трогаем.
Что касается части с novelty - пытаемся улучшить через catboost.
Пытаемся обучиться на silver_set (делая по нему split) - так как вручную сделанный golden мал для обучения, он чисто для теста.

In [63]:
#Импорты
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity

from catboost import CatBoostClassifier, Pool

SILVER_PATH = PREPARED_DIR / "lenta_silver_set_llm.csv"
HUMAN_GOLDEN_PATH = REFERENCE_PATH

BASELINE_PRED_PATH = OUTPUT_PATH
MODELS_DIR = ARTIFACTS_DIR / "models"
MODEL_PRED_PATH = PREPARED_DIR / "lenta_model_predictions.csv"
CATBOOST_MODEL_PATH = MODELS_DIR / "models/catboost_novelty_model.cbm"

EMBEDDINGS_CACHE_PATH = ARTIFACTS_DIR / "bge_m3_embeddings_by_id.pkl"

In [64]:
#Вспомогательное
VALID_LABELS = ["duplicate", "minor", "significant"]


def extract_numbers(text: str) -> set[str]:
    if pd.isna(text):
        return set()
    return set(re.findall(r"\d+(?:[.,]\d+)?", str(text)))


def token_set(text: str) -> set[str]:
    if pd.isna(text):
        return set()
    text = str(text).lower()
    return set(re.findall(r"[а-яa-zё0-9]{3,}", text))


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / max(len(a | b), 1)


def get_embedding(news_id, embeddings_by_id):
    emb = embeddings_by_id.get(news_id)
    if emb is None:
        raise KeyError(f"No embedding for news_id={news_id}")
    return emb

In [68]:
#Доработка Dataset, чтобы помимо текущего текста, учитывались прошлые тексты в кластере.
def build_novelty_feature_table(df: pd.DataFrame, embeddings_by_id: dict) -> pd.DataFrame:
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"])
    df = df.sort_values(["cluster_id", "published_at", "news_id"])

    rows = []

    for cluster_id, group in df.groupby("cluster_id"):
        group = group.sort_values(["published_at", "news_id"]).reset_index(drop=True)

        for i in range(len(group)):
            current = group.iloc[i]

            # Первая новость в кластере не имеет novelty_label.
            if i == 0:
                continue

            label = current.get("novelty_label", None)
            if label not in VALID_LABELS:
                continue

            previous = group.iloc[:i]

            current_emb = get_embedding(current["news_id"], embeddings_by_id).reshape(1, -1)
            previous_embs = np.vstack([
                get_embedding(row["news_id"], embeddings_by_id)
                for _, row in previous.iterrows()
            ])

            sims = cosine_similarity(current_emb, previous_embs)[0]
            top_sims = np.sort(sims)[::-1]

            prev_last = previous.iloc[-1]

            current_title_tokens = token_set(current["title"])
            previous_title_tokens = [token_set(t) for t in previous["title"]]

            current_text_tokens = token_set(current["text"])
            previous_text_tokens = [token_set(t) for t in previous["text"]]

            current_numbers = extract_numbers(current["text"])
            previous_numbers_union = set().union(*[extract_numbers(t) for t in previous["text"]])

            rows.append({
                "news_id": current["news_id"],
                "cluster_id": cluster_id,
                "topic": current.get("topic", "unknown"),

                "position_in_cluster": i,
                "cluster_size_so_far": len(previous),

                "days_since_previous": (
                    current["published_at"] - prev_last["published_at"]
                ).total_seconds() / 86400,

                "days_since_cluster_start": (
                    current["published_at"] - group.iloc[0]["published_at"]
                ).total_seconds() / 86400,

                "max_prev_similarity": float(np.max(sims)),
                "mean_prev_similarity": float(np.mean(sims)),
                "min_prev_similarity": float(np.min(sims)),
                "top2_mean_similarity": float(np.mean(top_sims[:2])) if len(top_sims) >= 2 else float(top_sims[0]),
                "top3_mean_similarity": float(np.mean(top_sims[:3])) if len(top_sims) >= 3 else float(np.mean(top_sims)),
                "last_prev_similarity": float(sims[-1]),

                "title_jaccard_max": max(
                    [jaccard(current_title_tokens, s) for s in previous_title_tokens],
                    default=0.0,
                ),
                "text_jaccard_max": max(
                    [jaccard(current_text_tokens, s) for s in previous_text_tokens],
                    default=0.0,
                ),

                "shared_numbers_count": len(current_numbers & previous_numbers_union),
                "new_numbers_count": len(current_numbers - previous_numbers_union),

                "title_length": len(str(current["title"])),
                "text_length": len(str(current["text"])),

                "target": label,
            })

    return pd.DataFrame(rows)

def make_embedding_source_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "model_text" in df.columns:
        text_col = "model_text"
    elif "text" in df.columns:
        text_col = "text"
    else:
        raise KeyError("Expected either 'model_text' or 'text' column")

    result = df[["news_id", text_col]].copy()
    result = result.rename(columns={text_col: "model_text"})
    result["model_text"] = result["model_text"].fillna("").astype(str)

    return result

In [69]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

if EMBEDDINGS_CACHE_PATH.exists():
    with open(EMBEDDINGS_CACHE_PATH, "rb") as f:
        embeddings_by_id = pickle.load(f)
    print(f"Loaded embeddings: {len(embeddings_by_id)}")
else:
    embeddings_by_id = {}
    print("No embeddings cache found, starting from empty dict")

No embeddings cache found, starting from empty dict


In [70]:
silver_df = pd.read_csv(SILVER_PATH)
golden_df = pd.read_csv(HUMAN_GOLDEN_PATH)
baseline_pred_df = pd.read_csv(BASELINE_PRED_PATH)

all_for_embeddings = pd.concat(
    [
        make_embedding_source_df(silver_df),
        make_embedding_source_df(golden_df),
        make_embedding_source_df(baseline_pred_df),
    ],
    ignore_index=True,
)

all_for_embeddings = all_for_embeddings.drop_duplicates("news_id")

missing_df = all_for_embeddings[
    ~all_for_embeddings["news_id"].isin(embeddings_by_id.keys())
].copy()

print("Need embeddings for:", len(missing_df))

Need embeddings for: 3176


In [71]:
def encode_texts_batched(texts, model, batch_size=32):
    embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch = texts[start:start + batch_size]

        batch_embeddings = model.encode(
            batch,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        embeddings.append(batch_embeddings)

    if not embeddings:
        return np.empty((0, 0))

    return np.vstack(embeddings)

In [73]:
if len(missing_df) > 0:
    texts = missing_df["model_text"].astype(str).tolist()
    ids = missing_df["news_id"].tolist()

    new_embeddings = encode_texts_batched(
        texts,
        model=baseline.model,   # замени на имя твоей bge-m3 модели
        batch_size=16,
    )

    for news_id, emb in zip(ids, new_embeddings):
        embeddings_by_id[news_id] = emb.astype(np.float32)

    with open(EMBEDDINGS_CACHE_PATH, "wb") as f:
        pickle.dump(embeddings_by_id, f)

    print(f"Saved embeddings cache: {len(embeddings_by_id)}")
else:
    print("All required embeddings already exist")

  0%|          | 0/199 [00:00<?, ?it/s]

Saved embeddings cache: 3176


In [74]:
required_ids = set(all_for_embeddings["news_id"])
available_ids = set(embeddings_by_id.keys())
missing_ids = required_ids - available_ids

print("Required:", len(required_ids))
print("Available:", len(available_ids))
print("Missing:", len(missing_ids))

assert len(missing_ids) == 0, "Some required embeddings are still missing"

Required: 3176
Available: 3176
Missing: 0


In [75]:
#Разделение silver set на train/test
silver_df = pd.read_csv(SILVER_PATH)

train_features = build_novelty_feature_table(
    silver_df,
    embeddings_by_id=embeddings_by_id,  # используй уже готовый dict из baseline
)

print(train_features.shape)
train_features["target"].value_counts()

(266, 20)


target
significant    206
minor           51
duplicate        9
Name: count, dtype: int64

In [76]:
feature_cols = [
    "position_in_cluster",
    "cluster_size_so_far",
    "days_since_previous",
    "days_since_cluster_start",
    "max_prev_similarity",
    "mean_prev_similarity",
    "min_prev_similarity",
    "top2_mean_similarity",
    "top3_mean_similarity",
    "last_prev_similarity",
    "title_jaccard_max",
    "text_jaccard_max",
    "shared_numbers_count",
    "new_numbers_count",
    "title_length",
    "text_length",
]

cat_features = []  # topic пока не добавляем, чтобы не переобучаться на темы
target_col = "target"
group_col = "cluster_id"

In [77]:
#Делим на train/test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(
    splitter.split(
        train_features,
        y=train_features[target_col],
        groups=train_features[group_col],
    )
)

train_part = train_features.iloc[train_idx].copy()
val_part = train_features.iloc[val_idx].copy()

X_train = train_part[feature_cols]
y_train = train_part[target_col]

X_val = val_part[feature_cols]
y_val = val_part[target_col]

In [ ]:
#Обучаем catboost
X_train_np = X_train.to_numpy(dtype=np.float32)
X_val_np = X_val.to_numpy(dtype=np.float32)

model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=500,
    depth=5,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=50,
    #auto_class_weights="Balanced", #плохо работает, даёт много фиктивных дубликатов
)

model.fit(
    X_train_np,
    y_train,
    eval_set=(X_val_np, y_val),
    use_best_model=True,
)

0:	learn: 0.5386853	test: 0.5059602	best: 0.5059602 (0)	total: 1.52ms	remaining: 761ms
50:	learn: 0.8446387	test: 0.3483082	best: 0.5059602 (0)	total: 49.4ms	remaining: 435ms
100:	learn: 0.9015484	test: 0.3546840	best: 0.5059602 (0)	total: 96.6ms	remaining: 382ms
150:	learn: 0.9372891	test: 0.3540448	best: 0.5059602 (0)	total: 147ms	remaining: 340ms
200:	learn: 0.9466637	test: 0.3483082	best: 0.5059602 (0)	total: 200ms	remaining: 298ms
250:	learn: 0.9760984	test: 0.3483082	best: 0.5059602 (0)	total: 254ms	remaining: 252ms
300:	learn: 0.9820846	test: 0.3483082	best: 0.5059602 (0)	total: 305ms	remaining: 202ms
350:	learn: 0.9900772	test: 0.3483082	best: 0.5059602 (0)	total: 356ms	remaining: 151ms
400:	learn: 0.9920624	test: 0.2997379	best: 0.5059602 (0)	total: 404ms	remaining: 99.8ms
450:	learn: 0.9940471	test: 0.2702792	best: 0.5059602 (0)	total: 453ms	remaining: 49.2ms
499:	learn: 0.9960316	test: 0.2702792	best: 0.5059602 (0)	total: 501ms	remaining: 0us

bestTest = 0.5059602421
bestIte

In [97]:
#Смотрим что получилось, сохраняем
val_pred = model.predict(X_val_np).ravel()

print(classification_report(y_val, val_pred, digits=4))
print("macro_f1:", f1_score(y_val, val_pred, average="macro"))
print (CATBOOST_MODEL_PATH)

import joblib
from pathlib import Path

CATBOOST_JOBLIB_PATH = MODELS_DIR / "catboost_novelty_model.joblib"
CATBOOST_JOBLIB_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, CATBOOST_JOBLIB_PATH)
#model.save_model(str(CATBOOST_MODEL_PATH), format="cbm")

              precision    recall  f1-score   support

   duplicate     0.1429    0.5000    0.2222         2
       minor     0.0000    0.0000    0.0000         6
 significant     0.8571    0.7895    0.8219        38

    accuracy                         0.6739        46
   macro avg     0.3333    0.4298    0.3480        46
weighted avg     0.7143    0.6739    0.6886        46

macro_f1: 0.3480466768138
E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\models\catboost_novelty_model.cbm


['E:\\ML\\Projects\\Git\\news-flow-analysis\\data\\artifacts\\models\\catboost_novelty_model.joblib']

In [98]:
def predict_novelty_with_model(pred_df: pd.DataFrame, embeddings_by_id: dict, model) -> pd.DataFrame:
    pred_df = pred_df.copy()
    pred_df["published_at"] = pd.to_datetime(pred_df["published_at"])
    pred_df = pred_df.sort_values(["cluster_id", "published_at", "news_id"])

    output_rows = []

    for cluster_id, group in pred_df.groupby("cluster_id"):
        group = group.sort_values(["published_at", "news_id"]).reset_index(drop=True)

        for i in range(len(group)):
            current = group.iloc[i].copy()

            if i == 0:
                predicted_label = ""
                needs_review = False
                comment = "model: first item in cluster"
            else:
                tmp_df = group.iloc[:i + 1].copy()
                tmp_features = build_novelty_feature_table(tmp_df, embeddings_by_id)

                # build_novelty_feature_table ожидает target, поэтому для inference проще берём последнюю строку,
                # если она построилась. Если target отсутствует, добавим фиктивный.
                if tmp_features.empty:
                    predicted_label = ""
                    needs_review = True
                    comment = "model: no features"
                else:
                    x = tmp_features.iloc[[-1]][feature_cols]
                    proba = model.predict_proba(x)[0]
                    classes = list(model.classes_)

                    best_idx = int(np.argmax(proba))
                    predicted_label = classes[best_idx]

                    sorted_proba = np.sort(proba)[::-1]
                    margin = sorted_proba[0] - sorted_proba[1] if len(sorted_proba) > 1 else 1.0

                    needs_review = margin < 0.15
                    comment = f"model: catboost novelty, confidence={sorted_proba[0]:.3f}, margin={margin:.3f}"

            output_rows.append({
                "news_id": current["news_id"],
                "published_at": current["published_at"],
                "topic": current.get("topic", ""),
                "title": current["title"],
                "text": current["text"],
                "cluster_id": current["cluster_id"],
                "novelty_label": predicted_label,
                "comment": comment,
                "needs_review": needs_review,
            })

    return pd.DataFrame(output_rows)

In [99]:
baseline_pred_df = pd.read_csv(BASELINE_PRED_PATH)

# Для inference временно добавим фиктивные labels,
# чтобы feature-builder не отбрасывал строки.
baseline_for_inference = baseline_pred_df.copy()
baseline_for_inference["novelty_label"] = "minor"

model_pred_df = predict_novelty_with_model(
    baseline_for_inference,
    embeddings_by_id=embeddings_by_id,
    model=model,
)

model_pred_df.to_csv(MODEL_PRED_PATH, index=False)
model_pred_df.head()

,news_id,published_at,topic,title,text,cluster_id,novelty_label,comment,needs_review
0,88522,2004-03-01,Мир,Расширение НАТО случится досрочно,Церемония принятия в НАТО новых членов состоит...,baseline_cluster_00000,,model: first item in cluster,False
1,88525,2004-03-01,Мир,Подозреваемый в убийстве диспетчера Skyguide г...,"Немецкие адвокаты, которые представляют интере...",baseline_cluster_00001,,model: first item in cluster,False
2,88528,2004-03-01,Мир,Правительственный совет Ирака одобрил проект в...,Члены Правительственного совета Ирака приняли ...,baseline_cluster_00002,,model: first item in cluster,False
3,88534,2004-03-01,Мир,Совет безопасности ООН одобрил отправку миротв...,Члены Совета безопасности ООН в воскресенье ве...,baseline_cluster_00003,,model: first item in cluster,False
4,88547,2004-03-01,Мир,Совет безопасности ООН собрался для обсуждения...,Совет безопасности ООН начал экстренное совеща...,baseline_cluster_00003,significant,"model: catboost novelty, confidence=0.335, mar...",True


In [100]:
#Проверка после обучения
print("Validation labels:")
print(pd.Series(y_val).value_counts())

print("Predicted labels:")
print(pd.Series(val_pred).value_counts())

print(classification_report(y_val, val_pred, digits=4))

Validation labels:
target
significant    38
minor           6
duplicate       2
Name: count, dtype: int64
Predicted labels:
significant    35
duplicate       7
minor           4
Name: count, dtype: int64
              precision    recall  f1-score   support

   duplicate     0.1429    0.5000    0.2222         2
       minor     0.0000    0.0000    0.0000         6
 significant     0.8571    0.7895    0.8219        38

    accuracy                         0.6739        46
   macro avg     0.3333    0.4298    0.3480        46
weighted avg     0.7143    0.6739    0.6886        46



In [101]:
model_pred_df = predict_novelty_with_model(
    baseline_pred_df,
    embeddings_by_id=embeddings_by_id,
    model=model,
)

model_pred_df.to_csv(MODEL_PRED_PATH, index=False)

model_pred_df["novelty_label"].value_counts(dropna=False)

novelty_label
               2607
significant     447
duplicate        73
minor            49
Name: count, dtype: int64

In [102]:
human_df = pd.read_csv(HUMAN_GOLDEN_PATH)
baseline_eval_df = pd.read_csv(BASELINE_PRED_PATH)
model_eval_df = pd.read_csv(MODEL_PRED_PATH)

In [103]:
from sklearn.metrics import recall_score, precision_score

def evaluate_novelty_by_news_id(reference_df, candidate_df, name="candidate"):
    reference = reference_df.copy()
    candidate = candidate_df.copy()

    reference["novelty_label"] = reference["novelty_label"].fillna("")
    candidate["novelty_label"] = candidate["novelty_label"].fillna("")

    merged = reference[["news_id", "novelty_label"]].merge(
        candidate[["news_id", "novelty_label"]],
        on="news_id",
        how="inner",
        suffixes=("_ref", "_pred"),
    )

    eval_rows = merged[
        merged["novelty_label_ref"].isin(["duplicate", "minor", "significant"])
    ].copy()

    y_true = eval_rows["novelty_label_ref"]
    y_pred = eval_rows["novelty_label_pred"]

    important_true = y_true.eq("significant")
    important_pred = y_pred.eq("significant")

    result = {
        "name": name,
        "reference_rows": len(reference),
        "candidate_rows": len(candidate),
        "overlap_rows": len(merged),
        "novelty_eval_rows": len(eval_rows),
        "novelty_accuracy": accuracy_score(y_true, y_pred),
        "novelty_macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "important_precision": precision_score(important_true, important_pred, zero_division=0),
        "important_recall": recall_score(important_true, important_pred, zero_division=0),
        "important_f1": f1_score(important_true, important_pred, zero_division=0),
    }

    print(f"\n=== {name} ===")
    print(pd.Series(result))

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=["duplicate", "minor", "significant"],
        digits=4,
        zero_division=0,
    ))

    print("Confusion matrix:")
    display(pd.DataFrame(
        confusion_matrix(
            y_true,
            y_pred,
            labels=["duplicate", "minor", "significant"],
        ),
        index=["true_duplicate", "true_minor", "true_significant"],
        columns=["pred_duplicate", "pred_minor", "pred_significant"],
    ))

    return result, eval_rows

In [104]:
baseline_result, baseline_rows = evaluate_novelty_by_news_id(
    human_df,
    baseline_eval_df,
    name="baseline",
)

model_result, model_rows = evaluate_novelty_by_news_id(
    human_df,
    model_eval_df,
    name="catboost_model",
)


=== baseline ===
name                   baseline
reference_rows              121
candidate_rows             3176
overlap_rows                121
novelty_eval_rows            87
novelty_accuracy        0.45977
novelty_macro_f1        0.25053
important_precision    0.822222
important_recall       0.506849
important_f1           0.627119
dtype: object

Classification report:
              precision    recall  f1-score   support

   duplicate     0.1818    0.4000    0.2500         5
       minor     0.1429    0.1111    0.1250         9
 significant     0.8222    0.5068    0.6271        73

   micro avg     0.6349    0.4598    0.5333        87
   macro avg     0.3823    0.3393    0.3340        87
weighted avg     0.7151    0.4598    0.5535        87

Confusion matrix:


,pred_duplicate,pred_minor,pred_significant
true_duplicate,2,0,3
true_minor,1,1,5
true_significant,8,6,37



=== catboost_model ===
name                   catboost_model
reference_rows                    121
candidate_rows                   3176
overlap_rows                      121
novelty_eval_rows                  87
novelty_accuracy             0.471264
novelty_macro_f1             0.250739
important_precision          0.863636
important_recall             0.520548
important_f1                 0.649573
dtype: object

Classification report:
              precision    recall  f1-score   support

   duplicate     0.1429    0.4000    0.2105         5
       minor     0.2000    0.1111    0.1429         9
 significant     0.8636    0.5205    0.6496        73

   micro avg     0.6508    0.4713    0.5467        87
   macro avg     0.4022    0.3439    0.3343        87
weighted avg     0.7536    0.4713    0.5719        87

Confusion matrix:


,pred_duplicate,pred_minor,pred_significant
true_duplicate,2,2,1
true_minor,1,1,5
true_significant,11,2,38


In [105]:
comparison_df = pd.DataFrame([baseline_result, model_result])
comparison_df

,name,reference_rows,candidate_rows,overlap_rows,novelty_eval_rows,novelty_accuracy,novelty_macro_f1,important_precision,important_recall,important_f1
0,baseline,121,3176,121,87,0.459770,0.250530,0.822222,0.506849,0.627119
1,catboost_model,121,3176,121,87,0.471264,0.250739,0.863636,0.520548,0.649573


In [106]:
diff_df = (
    human_df[["news_id", "title", "novelty_label"]]
    .merge(
        baseline_eval_df[["news_id", "novelty_label"]],
        on="news_id",
        how="left",
        suffixes=("_human", "_baseline"),
    )
    .merge(
        model_eval_df[["news_id", "novelty_label"]],
        on="news_id",
        how="left",
    )
    .rename(columns={"novelty_label": "novelty_label_model"})
)

diff_df = diff_df[
    diff_df["novelty_label_human"].isin(["duplicate", "minor", "significant"])
].copy()

diff_df[
    diff_df["novelty_label_baseline"] != diff_df["novelty_label_model"]
][[
    "news_id",
    "title",
    "novelty_label_human",
    "novelty_label_baseline",
    "novelty_label_model",
]].head(30)

,news_id,title,novelty_label_human,novelty_label_baseline,novelty_label_model
1,88596,Изгнанный президент Гаити прилетел в Центральн...,significant,NaN,NaN
3,88827,На родине культа вуду введено чрезвычайное пол...,significant,NaN,NaN
9,88614,Наряд дагестанских пограничников еще раз уничт...,duplicate,duplicate,minor
11,88715,Уничтожившие Гелаева пограничники представлены...,significant,NaN,NaN
14,89300,ОПЕК временно отменила нефтяные квоты,significant,NaN,NaN
15,89751,"ОПЕК обеспокоена ростом цен на нефть, но снизи...",significant,NaN,NaN
17,88687,"Кредитный рейтинг ""ЮКОСа"" сделают еще негативнее",significant,NaN,NaN
20,88970,Вслед за Opportunity следы воды на Марсе нашел...,significant,NaN,NaN
23,88728,"""Транс Нафта"" продлила поставки газа в Белоруссию",significant,NaN,NaN
25,89180,СИБУР продал Лукашенко миллиард кубометров газа,significant,NaN,NaN
